In [0]:
import requests
import json
from pyspark.sql import Row
from pyspark.sql.window import Window
from pyspark.sql.functions import *

## Deduping Bronze boxscore

In [0]:
bronze_boxscore = spark.table("bronze.mlb_boxscore_raw")

## Flattening the bronze boxscore into batting and pitching tables

**Key Notes**
*   'players' = **dictionary** keyed by 'ID<personID>'
*   Every player has BOTH 'batting' and 'pitching' stats. One or both will be an empty dict '{}' if they didn't bat or pitch. It's not a null, just empty dict
*   Position players who never batted (hurt and removed before their first AB) will have an empty batting dict, which is different from "0 at-bats logged"

In [0]:
def extract_player_rows(boxscore_json: dict, game_pk: int) -> tuple[list, list]:
    """
    Given one game's boxscore JSON, return two lists of dicts: (batting_rows, pitching_rows)
    - one row per player per stat category, skipping players who have no stats in that category
    """
    batting_rows = []
    pitching_rows = []

    teams = boxscore_json.get("teams", {})

    for home_away in ("home", "away"):
        team_block = teams.get(home_away, {})
        team_info = team_block.get("team", {})
        team_id = team_info.get("id")
        team_name = team_info.get("name")

        players = team_block.get("players", {})

        for player_key, player_data in players.items():
            person = player_data.get("person", {})
            player_id = person.get("id")
            player_name = person.get("fullName")
            position = player_data.get("position", {}).get("abbreviation")

            stats = player_data.get("stats", {})
            batting_stats = stats.get("batting",{})
            pitching_stats = stats.get("pitching", {})

            # Only creating row if the player actually has stats in batting or pitching dictionaries.
            # empty dicts = "did not bat" / "did not pitch"

            if batting_stats:
                batting_rows.append({
                    "game_pk": game_pk,
                    "player_id": player_id,
                    "player_name": player_name,
                    "team_id": team_id,
                    "team_name": team_name,
                    "home_away": home_away,
                    "position": position,
                    "at_bats": batting_stats.get("atBats"),
                    "hits": batting_stats.get("hits"),
                    "runs": batting_stats.get("runs"),
                    "home_run": batting_stats.get("homeRuns"),
                    "rbi": batting_stats.get("rbi"),
                    "walks": batting_stats.get("baseOnBalls"),
                    "strikeouts": batting_stats.get("strikeOuts"),
                    "stolen_bases": batting_stats.get("stolenBases")
                })

            if pitching_stats:
                pitching_rows.append({
                    "game_pk": game_pk,
                    "player_id": player_id,
                    "player_name": player_name,
                    "team_id": team_id,
                    "team_name": team_name,
                    "home_away": home_away,
                    "innings_pitched": pitching_stats.get("inningsPitched"),
                    "hits_allowed": pitching_stats.get("hits"),
                    "runs_allowed": pitching_stats.get("runs"),
                    "earned_runs": pitching_stats.get("earnedRuns"),
                    "walks_allowed": pitching_stats.get("baseOnBalls"),
                    "strikeouts_pitched": pitching_stats.get("strikeOuts"),
                    "home_runs_allowed": pitching_stats.get("homeRuns"),
                    "pitches_thrown": pitching_stats.get("numberOfPitches")               
                })

    return batting_rows, pitching_rows


## Running flattening across all games in Silver's input

### Using the deduped bronze boxscore for batting and pitching

In [0]:
all_batting_rows = []
all_pitching_rows = []
parse_failures = []

for row in bronze_boxscore.collect():
    try:
        raw = json.loads(row.raw_json)
        batting, pitching = extract_player_rows(raw, row.game_pk)
        all_batting_rows.extend(batting)
        all_pitching_rows.extend(pitching)
    except (json.JSONDecodeError, KeyError, TypeError) as e:
        print(f"[WARN] failed to parse game_pk {row.game_pk}: {e}")
        parse_failures.append(row.game_pk)



print(f"Batting rows: {len(all_batting_rows)}")
print(f"Pitching rows: {len(all_pitching_rows)}")
print(f"Parse failures: {len(parse_failures)}")

In [0]:
batting_df = spark.createDataFrame([Row(**r) for r in all_batting_rows]) \
    .dropDuplicates(["game_pk", "player_id"]) \
    .withColumn("silver_processed_at", current_timestamp())


batting_games = batting_df.select("game_pk").distinct()

missing_games = (
    bronze_boxscore
    .select("game_pk")
    .join(batting_games, on="game_pk", how="left_anti")
)

missing_game_pks = [
    row.game_pk
    for row in missing_games.collect()
]

print(f"Missing games: {len(missing_game_pks)}")
print(missing_game_pks)

In [0]:
recovered_boxscores = []
base_url = "https://statsapi.mlb.com/api/v1"

for game_pk in missing_game_pks:
    try:
        url = f"{base_url}/game/{game_pk}/boxscore"
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()

        recovered_boxscores.append({
            "game_pk": game_pk,
            "raw_json": resp.text
        })

        print(f"[OK] recovered game_pk {game_pk}")

    except requests.RequestException as e:
        print(f"[WARN] failed to recover game_pk {game_pk}: {e}")
print(f"Recovered boxscores: {len(recovered_boxscores)}")

In [0]:
for row in recovered_boxscores:
    try:
        raw = json.loads(row["raw_json"])
        batting, pitching = extract_player_rows(raw, row["game_pk"])

        all_batting_rows.extend(batting)
        all_pitching_rows.extend(pitching)

    except (json.JSONDecodeError, KeyError, TypeError) as e:
        print(f"[WARN] failed to parse recovered game_pk {row['game_pk']}: {e}")

## Building Spark DataFrames and Writing into Silver tables

### Batting stats

In [0]:
if all_batting_rows:
    batting_df = spark.createDataFrame([Row(**r) for r in all_batting_rows]) \
        .dropDuplicates(["game_pk", "player_id"]) \
        .withColumn("silver_processed_at", current_timestamp())
    
    batting_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("silver.mlb_batting_stats")

    display(batting_df.limit(10))

### Pitching Stats

In [0]:
if all_pitching_rows:
    pitching_df = spark.createDataFrame([Row(**r) for r in all_pitching_rows]) \
        .dropDuplicates(["game_pk", "player_id"]) \
        .withColumn("silver_processed_at", current_timestamp())
    
    pitching_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("silver.mlb_pitching_stats")

    display(pitching_df.limit(10))


# Data Quality Checks

In [0]:
print("Distinct games in batting:", batting_df.select("game_pk").distinct().count())
print("Distinct games in pitching:", pitching_df.select("game_pk").distinct().count())
print("Null player_id count (batting):", batting_df.filter(col("player_id").isNull()).count())

## Checking for missing games (games with no stats in boxscore)

## Rebuilding Spark DataFrames and Writing into Silver tables with the recovered boxscores

## Batting Stats

if all_batting_rows:
    batting_df = spark.createDataFrame([Row(**r) for r in all_batting_rows]) \
        .dropDuplicates(["game_pk", "player_id"]) \
        .withColumn("silver_processed_at", current_timestamp())
    
    batting_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("silver.mlb_batting_stats")

    display(batting_df.limit(10))

## Pitching Stats

if all_pitching_rows:
    pitching_df = spark.createDataFrame([Row(**r) for r in all_pitching_rows]) \
        .dropDuplicates(["game_pk", "player_id"]) \
        .withColumn("silver_processed_at", current_timestamp())
    
    pitching_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("silver.mlb_pitching_stats")

    display(pitching_df.limit(10))

print(f"{pitching_df.count()} pitching rows")
print(f"{batting_df.count()} batting rows")

still_missing = (
    deduped_boxscore.select("game_pk")
    .join(batting_df.select("game_pk").distinct(), on="game_pk", how="left_anti")
)

missing_details = (
    spark.table("silver.mlb_schedule")
    .join(still_missing, on="game_pk", how="inner")
    .select("game_pk", "game_date", "home_team_id", "away_team_id", "game_status")
)

display(missing_details)